# Machine Learning LAB 5: Random Forests

Course 2024/25: *F. Chiariotti*

The notebook contains a simple learning task over which we will implement a **RANDOM FOREST**.

Complete all the **required code sections**.

### IMPORTANT for the exam:

The functions you might be required to implement in the exam will have the same signature and parameters as the ones in the labs

## Classification of Stayed/Churned Customers

The Customer Churn table contains information on all 3,758 customers from a Telecommunications company in California in Q2 2022. Companies are naturally interested in churn, i.e., in which users are likely to switch to another company soon to get a better deal, and which are more loyal customers.

The dataset contains three features:
- **Tenure in Months**: Number of months the customer has stayed with the company
- **Monthly Charge**: The amount charged to the customer monthly
- **Age**: Customer's age

The aim of the task is to predict if a customer will churn or not based on the three features.

---

## Import all the necessary Python libraries and load the dataset

### The Dataset
The dataset is a `.csv` file containing three input features and a label. Here is an example of the first 4 rows of the dataset: 

<center>

Tenure in Months | Monthly Charge | Age | Customer Status |
| -----------------| ---------------|-----|-----------------|
| 9 | 65.6 | 37 | 0 |
| 9 | -4.0 | 46 | 0 |
| 4 | 73.9 | 50 | 1 |
| ... | ... | ... | ... |

</center>

Customer Status is 0 if the customer has stayed with the company and 1 if the customer has churned.

In [5]:
import numpy as np
import pandas as pd
import random as rnd
from matplotlib import pyplot as plt
from sklearn import linear_model, preprocessing
from sklearn.model_selection import train_test_split

np.random.seed(1)

def load_dataset(filename):
    data_train = pd.read_csv(filename)
    #permute the data
    data_train = data_train.sample(frac=1).reset_index(drop=True) # shuffle the data
    X = data_train.iloc[:, 0:3].values # Get first two columns as the input
    Y = data_train.iloc[:, 3].values # Get the third column as the label
    Y = 2*Y-1 # Make sure labels are -1 or 1 (0 --> -1, 1 --> 1)
    return X,Y

# Load the dataset
X, Y = load_dataset('data/telecom_customer_churn_cleaned.csv')

We are going to differentiate (classify) between **class "1" (churned)** and **class "-1" (stayed)**

## Divide the data into training and test sets

In [6]:
# Compute the splits
m_training = int(0.75*X.shape[0])

# m_test is the number of samples in the test set (total-training)
m_test =  X.shape[0] - m_training
X_training =  X[:m_training]
Y_training =  Y[:m_training]
X_test =   X[m_training:]
Y_test =  Y[m_training:]

print("Number of samples in the train set:", X_training.shape[0])
print("Number of samples in the test set:", X_test.shape[0])
print("Number of churned users in test:", np.sum(Y_test==-1))
print("Number of loyal users in test:", np.sum(Y_test==1))

# Standardize the input matrix
# The transformation is computed on training data and then used on all the 3 sets
scaler = preprocessing.StandardScaler().fit(X_training) 

np.set_printoptions(suppress=True) # sets to zero floating point numbers < min_float_eps
X_training =  scaler.transform(X_training)
print ("Mean of the training input data:", X_training.mean(axis=0))
print ("Std of the training input data:",X_training.std(axis=0))

X_test =  scaler.transform(X_test)
print ("Mean of the test input data:", X_test.mean(axis=0))
print ("Std of the test input data:", X_test.std(axis=0))
print(Y_training)

Number of samples in the train set: 2817
Number of samples in the test set: 940
Number of churned users in test: 479
Number of loyal users in test: 461
Mean of the training input data: [-0.  0. -0.]
Std of the training input data: [1. 1. 1.]
Mean of the test input data: [0.0575483  0.05550169 0.0073833 ]
Std of the test input data: [0.98593187 0.97629659 1.00427583]
[-1  1  1 ...  1 -1 -1]


We will use **homogeneous coordinates** to describe all the coefficients of the model.

_Hint:_ The conversion can be performed with the function $hstack$ in $numpy$.

## Decision tree

Now **complete** the class *Tree* and all auxiliary functions. <br>

The input parameters to pass to the *id3_training* function are:
- $X$: the matrix of input features, one row for each sample
- $Y$: the vector of labels for the input features matrix X
- $max\_depth$: the maximum depth of the tree

In [19]:
class Tree:

    def __init__(self):  # defining the structure of a decision tree node
        self.idx = -1    # The index of the feature over which you split (no split: -1)
        self.thresh = 0  # The threshold value over which you split (<=: left, >: right)
        self.leaf = 0    # 1 if it is a leaf of class 1, -1 if it is a leaf of class -1, 0 if it is an internal node
        self.left = []   # Left subtree (empty if it is a leaf)
        self.right = []  # Right subtree (empty if it is a leaf)


    def entropy(left, right):
        H = 0
        tot_length = len(left) + len(right)
        left_prob = len(np.where(left > 0)[0]) / len(left)
        if (left_prob > 0):
            H -= len(left) * left_prob * np.log2(left_prob) / tot_length
        if (left_prob < 1):
            H -= len(left) * (1 - left_prob) * np.log2(1 - left_prob) / tot_length
        right_prob = len(np.where(right > 0)[0]) / len(right)
        if (right_prob > 0):
            H -= len(right) * right_prob * np.log2(right_prob) / tot_length
        if (right_prob < 1):
            H -= len(right) * (1 - right_prob) * np.log2(1 - right_prob) / tot_length
        return H

    def classify(self, x):
        ## TO DO: classify the point x (easy for leaves, you have to go down the tree if the node is internal)
        #Base case: check if the node is a leaf --> if it is a leaf, return the class label
        if self.leaf != 0: 
            return self.leaf # 1 or -1

        # Recursive case: Check the value of the feature x[self.idx]
        if x[self.idx] <= self.thresh:
            # Move to the left child
            return self.left.classify(x) #making it a recursive method: The method is called again on the left or right child.
                                        #This continues until a leaf node is reached, and the class label is returned.
        else:
            # Move to the right child
            return self.right.classify(x)
        
    def id3_training(self, X, Y, max_depth, printing): 
        # First we have to decide if a node (decision point) should become a leaf or we should continue splitting
        
        # Check if the node is a leaf (all nodes have the same label)
        if (np.max(Y) - np.min(Y) < 1e-3):
            self.leaf = np.max(Y)
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (all labels are the same over ' + str(len(Y)) + ' points)')
            return
        
        # If the maximum depth is 0, the node must be a leaf! This is picked based on major voting. 
        if (max_depth < 1):
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (maximum depth reached, ' + str(len(Y)) + ' points)')
            if (len(np.where(Y > 0)) > len(Y) / 2):
                self.leaf = 1
            else:
                self.leaf = -1
            return
        
        # If the maximum depth is not zero yet, and the node is not a leaf yet, we can go on splitting
        
        # Initialize these variables with values that are never gonna be reached
        best_idx = -1 
        best_thresh = -1
        best_entropy = 1e9
        
        ## TO DO: Find the best split: iterate over the features and threshold values! 
        for idx in range(X.shape[1]): #N.B. First we iterate over dimensions (shape[0] is the number of samples I have, while shape[1] is the number of features (dimensions) of each data point
            # Try thresholds as midpoints between unique sorted feature values
            unique_values = np.unique(X[:, idx]) # returns the sorted (in ascending order) unique elements from a specific column (feature) of a NumPy array X
            thresholds = (unique_values[:-1] + unique_values[1:]) / 2 #[:-1] selects all points except the last one while [1:] selects all points except the first one
            for thresh in thresholds:
                left = np.where(X[:, idx] <= thresh)[0] # indices of the training set that go on the left of the threshold 
                right = np.where(X[:, idx] > thresh)[0] # indices of the training set that go on the right of the threshold
                
                # Add an error signal if all points are put on the same side (not required but was in teacher's solution)
                if (len(left) == 0 or len(right) == 0):
                    print('error!',best_idx,thresh,values)
                    
                H = Tree.entropy(Y[left], Y[right]) # compute the entropy corresponding to this configuration of +1, -1 points in the two groups
                
                # Keep track of the best split
                if H < best_entropy:
                    best_entropy = H
                    best_idx = idx
                    best_thresh = thresh
                    
        if (best_idx == -1):
            # No valid features! The points are all identical --> if all points are identical, the entropy is already minimal and no slit will reduce it. 
            # Indeed, if you have no points on one side, the entropy is infinite (or undefined) hence for how big you initialized it, the initial value will always be better than infty
            # NOTICE: having all identical points (same X) doesn't necessarily mean that they all have the same label! You might have different labels but it might be impossible to put some points on the left/right of the threshold
            # If you never have H < best_entropy, the value best_idx never gets updated 
            # When no valid split is found, the node must become a leaf that represents the most likely class for the data at that point.
            self.leaf = np.sign(np.sum(Y))
            if (self.leaf == 0):
                self.leaf = 1
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (all inputs are the same over ' + str(len(Y)) + ' points)')
            return
        left_samples = np.where(X[:, best_idx] <= best_thresh)[0]
        right_samples = np.where(X[:, best_idx] > best_thresh)[0]
        if (printing):
            print('Remaining depth: ' + str(max_depth) + ', splitting ' + str(len(Y)) + ' elements into ' + str(len(left_samples)) + ' and ' + str(len(right_samples)) + ' over feature ' + str(best_idx))
        
        ## TO DO: run the next recursive step of ID3 over the left and right subtrees!
        # We do the split
        self.idx = best_idx
        self.thresh = best_thresh
        self.left = Tree()
        self.right = Tree()
        # For both the left and right tree we iterate the algorithm on right and left samples
        # NOTICE: now the max_depth is decreased by one
        self.left.id3_training(X[left_samples, :], Y[left_samples], max_depth - 1, printing)
        self.right.id3_training(X[right_samples, :], Y[right_samples], max_depth - 1, printing)


    def extra_training(self, X, Y, max_depth, printing):
        #Again we fist check if we should split or not
        
        # Check if the node is a leaf (all nodes have the same label)
        if (np.max(Y) - np.min(Y) < 1e-3):
            self.leaf = np.max(Y)
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (all labels are the same over ' + str(len(Y)) + ' points)')
            return
            
        # If the maximum depth is 0, the node must be a leaf!
        if (max_depth < 1):
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (maximum depth reached, ' + str(len(Y)) + ' points)')
            if (len(np.where(Y > 0)) > len(Y) / 2):
                self.leaf = 1
            else:
                self.leaf = -1
            return

        #If is not a leaf, or we have not reached maximum depth yet, we go on splitting
        # Find the best split: iterate over features
        best_idx = -1
        best_thresh = -1
        best_entropy = 1e9
        ## TO DO: Iterate over the features (remember, the threshold value is random)! 
        for idx in range(X.shape[1]): 
            values = X[:, idx] # I select the current training data's feature
            #NB: you also need to check if the values are all the same (hence len(unique_values) = 1)
            values = np.unique(values) #This is needed only for len(values) later to do its job 
            # Randomize the split!
            if (len(values) > 1):
                thresh = rnd.uniform(np.min(values), np.max(values))
                left = np.where(X[:, idx] <= thresh)[0]
                right = np.where(X[:, idx] > thresh)[0]
                H = Tree.entropy(left, right)
                if (H < best_entropy):
                    best_idx = idx
                    best_thresh = thresh
                    best_entropy = H
            
        if (best_idx == -1):
            # No valid features! The points are all identical
            self.leaf = np.sign(np.sum(Y))
            if (self.leaf == 0):
                self.leaf = 1
            if (printing):
                print('Remaining depth: ' + str(max_depth) + ', leaf node (all inputs are the same over ' + str(len(Y)) + ' points)')
            return
        left_samples = np.where(X[:, best_idx] <= best_thresh)[0]
        right_samples = np.where(X[:, best_idx] > best_thresh)[0]
        if (printing):
            print('Remaining depth: ' + str(max_depth) + ', splitting ' + str(len(Y)) + ' elements into ' + str(len(left_samples)) + ' and ' + str(len(right_samples)) + ' over feature ' + str(best_idx))
        
        self.idx = best_idx
        self.thresh = best_thresh
        self.left = Tree()
        self.right = Tree()
        self.left.extra_training(X[left_samples, :], Y[left_samples], max_depth - 1, printing)
        self.right.extra_training(X[right_samples, :], Y[right_samples], max_depth - 1, printing)
        
        

### 1. **The `Tree` Class**

This code defines a class `Tree` that can be used as part of a decision tree structure, commonly used in machine learning for classification tasks. Here's what the attributes represent:

- **`self.idx`**: 
  - Represents the feature index on which the node splits the data.
  - If there's no split (the node is a leaf), the index remains `-1`.
  
- **`self.thresh`**:
  - The threshold value for splitting the data.
  - Typically, this means:
    - Values less than or equal to `self.thresh` go to the **left** subtree.
    - Values greater than `self.thresh` go to the **right** subtree.

- **`self.leaf`**: 
  - Indicates if the node is a leaf, and if so, what class it represents:
    - `1` → The leaf corresponds to class `+1`.
    - `-1` → The leaf corresponds to class `-1`.
    - `0` → The node is an **internal node** and has left and right children.

- **`self.left`** and **`self.right`**:
  - Represent the **left subtree** and the **right subtree**.
  - If a node is a leaf, these lists remain empty (`[]`).

This structure effectively builds a binary decision tree where nodes can either be:
1. **Internal nodes** (containing splits),
2. **Leaf nodes** (with a classification).

---

### 2. **Entropy Method**

The purpose of the `entropy` method is to calculate the **entropy** of a split into left and right partitions of the dataset. Entropy is a measure of impurity or uncertainty in a set of data points, used to decide how good a split is. 

#### Input:
- `left` and `right`: Arrays containing data split into two subsets.

#### Steps:
1. **Total Size of the Split**:
   ```python
   tot_length = len(left) + len(right)
   ```
   - `tot_length` is the total number of data points in both subsets (left + right).

2. **Calculate Left Probability**:
   ```python
   left_prob = len(np.where(left > 0)[0]) / len(left)
   ```
   - Here, `np.where(left > 0)[0]` finds the **indices of data points that are greater than 0**.
   - The length of these indices divided by `len(left)` gives the fraction of positive points in the left subset.

3. **Left Entropy Contribution**:
   ```python
   if (left_prob > 0):
       H -= len(left) * left_prob * np.log2(left_prob) / tot_length
   if (left_prob < 1):
       H -= len(left) * (1 - left_prob) * np.log2(1 - left_prob) / tot_length
   ```
   - The entropy for the left split is computed based on:
     - \( -p \log_2(p) \) → If the probability `p` is positive.
     - \( -(1-p) \log_2(1-p) \) → If the probability `p` is not 1 (to account for the negative class).
   - Each term is weighted by the size of the subset relative to the total number of points.

4. **Right Probability and Entropy**:
   - The same calculations are repeated for the right subset:
     ```python
     right_prob = len(np.where(right > 0)[0]) / len(right)
     ```
   - Then contributions to `H` are made similarly.

5. **Return the Total Entropy**:
   ```python
   return H
   ```
   - `H` now represents the weighted sum of the entropy for both the left and right subsets.

---

### Key Points About Entropy:
- If a subset is perfectly pure (all points belong to the same class), its entropy is zero.
- If a subset is maximally uncertain (equal number of both classes), its entropy is maximum.

The `entropy` method calculates a single number (H) that quantifies the uncertainty of splitting the data into two groups. This number can then be used to decide **where to split** a feature in a decision tree.

---

### 3. **The `classify` Method (TO DO)**
The `classify` method, while not yet implemented, will likely work as follows:
- Given a data point \(x\), the method will traverse the decision tree.
- Starting from the root, it will:
  1. Check if the node is a leaf:
     - Return the class label stored in `self.leaf` if it is.
  2. If the node is an internal node:
     - Compare the feature value in `x` against the threshold `self.thresh`.
     - Traverse the **left** subtree if the value is \(\leq \text{self.thresh}\).
     - Traverse the **right** subtree if the value is \( > \text{self.thresh} \).

The traversal will continue recursively until a leaf node is reached.

Good question! The `classify` method is defined as:

```python
def classify(self, x):
    ## TO DO: classify the point x ...
```

Here, the method receives **two parameters**: `self` and `x`.

1. **`self`**:
   - `self` is automatically passed when the method is called on an instance of the `Tree` class.
   - It allows the method to access attributes (like `self.idx`, `self.thresh`, `self.left`, `self.right`, and `self.leaf`) and other methods belonging to the current instance.
   - This is required in Python for instance methods. Without `self`, Python wouldn't know which instance of the `Tree` class the method is being invoked on.

2. **`x`**:
   - `x` is the actual input data point you are trying to classify.
   - `x` will typically be a feature vector (e.g., a list or NumPy array with numerical values).

---

### Why Not Call Only `x`?

If you tried to write `classify(x)` without `self` inside the class, Python would throw an error:

```
TypeError: classify() takes 1 positional argument but 2 were given
```

This happens because Python **always passes the instance of the class (`self`)** as the first argument when you call an instance method.

For example:
```python
tree = Tree()
tree.classify(x)
```

Internally, Python translates `tree.classify(x)` into:

```python
Tree.classify(tree, x)
```

Here, `tree` is passed as `self`, and `x` is passed as the second argument.

---

### The Purpose of `self`
You need `self` to access:
- Whether the current node (`self`) is a leaf or internal node.
- Its threshold value (`self.thresh`).
- Its split children (`self.left` and `self.right`).

For example, if the `classify` method were fully implemented, you'd do something like this:

```python
def classify(self, x):
    if self.leaf != 0:  # If this is a leaf node
        return self.leaf  # Return the class label

    # If this is not a leaf, decide to go left or right
    if x[self.idx] <= self.thresh:
        return self.left.classify(x)  # Recursive call to left subtree
    else:
        return self.right.classify(x)  # Recursive call to right subtree
```

Here:
- `self.leaf` tells you whether the current node is a leaf.
- `self.idx` and `self.thresh` tell you how the split is defined.
- `self.left` and `self.right` allow you to go deeper in the tree.

Without `self`, you could not access these attributes, and the method wouldn't know how to behave for the current node.

---

### In Summary
- `self` refers to the current instance of the class and is **mandatory** for all instance methods in Python.
- It allows the method to access the tree's structure and values to perform its task.
- `x` is the argument you explicitly provide to classify a new data point. 

Certainly! I can help you complete the `classify` method. The goal of the method is to classify a data point \( x \) by traversing the decision tree.

Here’s the logic for implementing the method:

1. **Base case** (leaf node):
   - If the current node is a leaf (`self.leaf != 0`), return the class label stored in `self.leaf` (which is `1` or `-1`).

2. **Recursive case** (internal node):
   - Check the value of the feature `x[self.idx]` (the feature corresponding to `self.idx`).
   - If the value is **less than or equal** to `self.thresh`, move to the **left subtree** (`self.left`).
   - If the value is **greater** than `self.thresh`, move to the **right subtree** (`self.right`).

3. Recursively call the `classify` method on the correct child (left or right subtree).

Here’s the code:

---

### Completed `classify` Method:
```python
def classify(self, x):
    # Base case: If this is a leaf, return the class label
    if self.leaf != 0:
        return self.leaf  # 1 or -1
    
    # Recursive case: Check the value of the feature x[self.idx]
    if x[self.idx] <= self.thresh:
        # Move to the left child
        return self.left.classify(x)
    else:
        # Move to the right child
        return self.right.classify(x)
```

---

### Explanation:
1. **`self.leaf != 0`**:
   - If the node is a leaf, `self.leaf` will contain `1` or `-1`.  
   - We return the class label immediately, as this is the prediction for the input \( x \).

2. **`x[self.idx]`**:
   - Accesses the value of feature \( \text{idx} \) in the input data point \( x \).
   - `self.idx` tells us **which feature** the current node is splitting on.

3. **Threshold comparison (`<= self.thresh`)**:
   - If the feature value \( x[self.idx] \) is **less than or equal** to `self.thresh`, we proceed to the **left subtree** (`self.left`).
   - Otherwise, we proceed to the **right subtree** (`self.right`).

4. **Recursive call**:
   - The method is called again on the left or right child.
   - This continues until a leaf node is reached, and the class label is returned.

---

### Usage Example:
Here's an example of how this method can be used in a tree:

```python
# Create a simple tree structure manually for testing
root = Tree()
root.idx = 0  # Split on feature 0
root.thresh = 5

# Define left child (a leaf)
root.left = Tree()
root.left.leaf = 1  # Class 1

# Define right child (a leaf)
root.right = Tree()
root.right.leaf = -1  # Class -1

# Input data point
x = [3]  # Feature 0 has value 3
print(root.classify(x))  # Output: 1 (since 3 <= 5, move to left)

x = [8]  # Feature 0 has value 8
print(root.classify(x))  # Output: -1 (since 8 > 5, move to right)
```

---

### Output:
```
1
-1
```

---

### Summary
The `classify` method traverses the tree recursively until it reaches a leaf node, at which point it returns the class label. The method relies on the current node's splitting condition (`self.thresh` and `self.idx`) to decide whether to go left or right.

Let me tackle your question and clarify the roles of the checks in **`classify`** and **`id3_training`**.

---

### Why do we check if a node is a leaf in `classify`?

**In `classify`:**  
The method **recursively traverses** the decision tree to classify a data point \( x \). When traversing, at some point, we will reach a **leaf node**. At this point, there are **no children** (subtrees) to go further, so we **stop recursion and return the class label** stored in the leaf.

Without checking for `self.leaf != 0`, the method wouldn't know when to stop traversal, and trying to call the method on `self.left` or `self.right` for a leaf would cause an error, as those attributes aren't valid trees when `self` is a leaf.

So **checking for leaf nodes** in `classify` is *necessary* because the node might have been set as a leaf by the **`id3_training` method** during the training phase.

---

### Why do we check for leaf nodes again in `id3_training`?

**In `id3_training`:**  
The ID3 algorithm determines if a node is a **leaf node** as part of the **training process**, based on one of these conditions:
1. All points \( Y \) have the **same label** → There is no reason to split further.
2. The **maximum depth** of the tree has been reached.

At this point, the node's `self.leaf` attribute is explicitly set to \( 1 \) or \( -1 \), and recursion stops.

The role of `id3_training` is to construct the tree and set appropriate splits, but during inference (when we classify points), we *must* check if a node is a leaf to determine when to stop traversing.

---

### Summary: Why check in both `classify` and `id3_training`?

- `id3_training` sets up whether a node becomes a leaf (during the training phase).
- `classify` checks for leaf nodes **during the prediction phase** so it knows when to stop traversing.

Without this redundancy, the two methods wouldn't achieve their goals independently.

---

### Notes:
1. **Iterate over features**: We loop over all features in `X` and find the best split (feature and threshold) that minimizes the entropy.
2. **Thresholds**: Midpoints between unique feature values are considered as possible thresholds for splits.
3. **Recursion**: After determining the split, the `id3_training` method recursively calls itself on the left and right subsets of the data.

---

### Implementing **`extra_training`**:
The `extra_training` method differs from `id3_training` because the threshold is selected **randomly** instead of testing all possible thresholds.


### Key Difference:
- **`id3_training`** searches for the **best split**.
- **`extra_training`** chooses splits randomly.

---

Now we use the implementation to learn a model from the training data directly. Set a maximum depth of 12.

In [212]:
single_tree = Tree()
single_tree.id3_training(X_training, Y_training, 12, True)

train_loss = 0
for i in range(len(Y_training)):
    predicted = single_tree.classify(X_training[i, :])
    if (Y_training[i] != predicted):
        train_loss += 1 / len(Y_training)
print('Training loss: ' + str(train_loss))

test_loss = 0
for i in range(len(Y_test)):
    predicted = single_tree.classify(X_test[i, :])
    if (Y_test[i] != predicted):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Remaining depth: 12, splitting 2817 elements into 464 and 2353 over feature 0
Remaining depth: 11, leaf node (all labels are the same over 464 points)
Remaining depth: 11, splitting 2353 elements into 467 and 1886 over feature 1
Remaining depth: 10, splitting 467 elements into 232 and 235 over feature 0
Remaining depth: 9, splitting 232 elements into 21 and 211 over feature 1
Remaining depth: 8, splitting 21 elements into 17 and 4 over feature 1
Remaining depth: 7, splitting 17 elements into 2 and 15 over feature 2
Remaining depth: 6, leaf node (all labels are the same over 2 points)
Remaining depth: 6, splitting 15 elements into 3 and 12 over feature 2
Remaining depth: 5, leaf node (all labels are the same over 3 points)
Remaining depth: 5, splitting 12 elements into 6 and 6 over feature 2
Remaining depth: 4, splitting 6 elements into 4 and 2 over feature 0
Remaining depth: 3, leaf node (all labels are the same over 4 points)
Remaining depth: 3, splitting 2 elements into 1 and 1 over 

#### **What Happens Without Calling `id3_training`?**
1. The `Tree` object is created with default values:
   - `self.idx = -1`: Indicates no feature index is set.
   - `self.thresh = 0`: No threshold is set.
   - `self.leaf = 0`: Indicates this node is not yet a leaf or doesn't represent a class.
   - `self.left = []` and `self.right = []`: The child nodes are empty lists, meaning there are no branches.

2. If you then call `classify`, the function will:
   - Check if `self.leaf != 0`. Since `self.leaf` is `0` by default, it proceeds to the recursive case.
   - Try to access `self.idx` and `self.thresh`, which remain uninitialized or meaningless for classification (`-1` and `0` respectively).
   - Attempt to call `classify` on `self.left` or `self.right`, but these are empty lists (`[]`), not valid `Tree` objects, leading to an error.

#### **Why Training (`id3_training`) Is Necessary**
The `id3_training` function is responsible for:
1. Splitting the data into subsets based on features and thresholds.
2. Assigning valid values to `self.idx` (feature index), `self.thresh` (threshold), and `self.leaf` (class label for leaf nodes).
3. Creating and populating `self.left` and `self.right` with new `Tree` objects, representing the left and right branches of the decision tree.

Without this step, the tree structure is undefined, and classification cannot proceed.

#### **What If You Skip Training?**
- **Expected Behavior:** The tree would not know how to classify inputs because it hasn't learned any decision boundaries or labels.
- **Error Likely:** When calling `classify`, the recursion would fail, as `self.left` and `self.right` are not properly set.

#### **What Happens After Training?**
When you explicitly call `single_tree.id3_training(X_training, Y_training, 12, True)`, the tree is constructed recursively. The root and subsequent nodes have valid `self.idx`, `self.thresh`, `self.leaf`, `self.left`, and `self.right`. This makes the tree functional for classification.


This overfits a lot! Let's try to create a random forest

In [20]:
class Forest:

    def __init__(self, trees, n_features, n_samples, max_depth): #Remember that __init__ is needed to define the attributes of the class
        self.forest = [] #List that stores individual decision trees 
        self.features = n_features #Specifies the number of features to use when building each tree
        self.max_depth = max_depth #Sets the maximum depth of each decision tree
        self.samples = n_samples #Determines the number of samples (data points) to use for training each tree
        for i in range(trees):
            self.forest.append(Tree())

    def classify(self, x):
        # Classify the point through a majority vote
        vote = 0
        for tree in self.forest:
            vote += tree.classify(x)
        return np.sign(vote)

    def train(self, X, Y):
        for tree in self.forest:
            X_train, Y_train = self.bag(X, Y)
            tree.id3_training(X_train, Y_train, self.max_depth, False)
            
    def bag(self, X, Y):
        ## TO DO: implement bagging! Sample data points with replacement --> this means that when you select a data point, you put it back into the pool before selecting the next one.
        features = X.shape[1]
        points = X.shape[0]
        bagged = rnd.choices(range(points), k = self.samples) #print(random.choices(mylist, weights = [10, 1, 1], k = 14))
                                                            #since we passed (range(points)) it will select point's indices
                                                            #CHOICES samples WITH REPLACEMENT
        X_bagged = X[bagged, :]
        # Remove features that are not part of the tree
        selected = rnd.sample(range(features), k = self.features) #SAMPLE samples WITHOUT REPLACEMENT
        for i in range(features):
            if i not in selected:
                X_bagged[:, i] = 0
        return X_bagged, Y[bagged]
        
        

While it’s common to use the same parameters for all trees in the forest for simplicity and theoretical consistency, it’s not mandatory. 
Random Forests are designed with these assumptions in mind, ensuring balanced diversity and stability in predictions.Varying parameters can increase diversity and potentially improve performance, but this requires more careful tuning and testing. The choice depends on your goals and the characteristics of your data.

Let us create a forest with 400 trees, with a maximum depth of 12. We can train each tree on the same number of points as the training set (using bagging to add some randomness). Since we do not have that many features, we keep all 3 features

In [21]:
forest = Forest(400, 3, X.shape[0], 12)
forest.train(X_training, Y_training)

test_loss = 0
for i in range(len(Y_test)):
    predicted = forest.classify(X_test[i, :])
    if (Y_test[i] * predicted <= 0):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Test loss: 0.294680851063829


Now we can try to see the effect of using Extra Trees. 

In [22]:
class ExtraTrees:

    def __init__(self, trees, n_features, n_samples, max_depth):
        self.forest = []
        self.features = n_features
        self.max_depth = max_depth
        self.samples = n_samples
        for i in range(trees):
            self.forest.append(Tree())

    def classify(self, x):
        vote = 0
        for tree in self.forest:
            vote += tree.classify(x)
        return np.sign(vote)

    def train(self, X, Y):
        for tree in self.forest:
            X_train, Y_train = self.bag(X, Y)
            tree.extra_training(X_train, Y_train, self.max_depth, False)
        
    def bag(self, X, Y):
        features = X.shape[1]
        points = X.shape[0]
        # Bagging: sample with replacement
        bagged = rnd.choices(range(points), k = self.samples)
        X_bagged = X[bagged, :]
        # Remove features that are not part of the tree
        selected = rnd.sample(range(features), k = self.features)
        for i in range(features):
            if i not in selected:
                X_bagged[:, i] = 0
        return X_bagged, Y[bagged]

Let us create an Extra Trees Classifer with 1000 trees, with a maximum depth of 12. We can train each tree on the same number of points as the training set (using bagging to add some randomness). Since we do not have that many features, we keep all 3 features

In [23]:
extra = ExtraTrees(1000, 3, X.shape[0], 12)
extra.train(X_training, Y_training)

test_loss = 0
for i in range(len(Y_test)):
    predicted = extra.classify(X_test[i, :])
    if (Y_test[i] * predicted <= 0):
        test_loss += 1 / len(Y_test)
print('Test loss: ' + str(test_loss))

Test loss: 0.2659574468085106


### **Extra Trees vs. ID3 Forest**
Extra Trees (Extremely Randomized Trees) and ID3 (Iterative Dichotomiser 3) forests are two different approaches to building decision tree ensembles. Whether one works better than the other depends on the dataset, but generally, **Extra Trees tends to perform better than an ID3-based forest**. Here’s why:

---

#### **1. Key Differences Between Extra Trees and ID3 Forest**
| **Aspect**              | **Extra Trees**                                         | **ID3 Forest**                                       |
|-------------------------|-------------------------------------------------------|----------------------------------------------------|
| **Splitting Criterion** | Random thresholds for splits.                         | Chooses the best threshold to maximize information gain. |
| **Feature Selection**   | Randomly selects a subset of features for each split. | Considers all features for splitting.             |
| **Randomness**          | High randomness in both feature selection and thresholds. | Lower randomness as splits are deterministic.     |
| **Overfitting Risk**    | Lower, due to high randomness.                        | Higher, especially for noisy or small datasets.   |

#### **2. Why Does Extra Trees Often Work Better?**

##### **a. Reduced Overfitting**
- **ID3 Forest:** Tends to overfit the training data because it always tries to find the "best" split by maximizing information gain. This can make the model too specific to the training set, especially with noisy or small datasets.
- **Extra Trees:** Introduces randomness in both feature selection and split thresholds. This reduces the likelihood of overfitting, making the model more robust to unseen data.

##### **b. Faster Training**
- **ID3 Forest:** Evaluates all possible thresholds for each feature to find the best split, which can be computationally expensive.
- **Extra Trees:** Randomly picks thresholds and doesn’t evaluate all possibilities, making training much faster.

##### **c. Increased Diversity Among Trees**
- **ID3 Forest:** Trees in the forest are less diverse because they all strive to find the "best" splits based on the same criterion.
- **Extra Trees:** Random splits introduce more variation among trees, leading to better ensemble performance as the trees make less correlated errors.

##### **d. Robustness to Noise**
- **ID3 Forest:** Overfits noisy data, as it seeks the perfect split that may capture noise patterns rather than general trends.
- **Extra Trees:** Random splits are less likely to fit noise, improving robustness.

#### **3. Situations Where ID3 Forest Might Work Better**
- **Small datasets:** If the dataset is small and relatively clean, the deterministic nature of ID3's splits may lead to better performance.
- **Feature importance:** If understanding feature importance is critical, ID3 forests are more interpretable as they select splits based on information gain.

#### **4. Empirical Evidence**
In most real-world scenarios:
- **Extra Trees outperform ID3 forests** due to better generalization, reduced overfitting, and faster training.
- ID3 forests may excel in highly structured datasets with limited noise, where deterministic splits provide an advantage.
  
### **Conclusion**
Extra Trees forests are generally more robust, faster, and better suited for large, noisy, or high-dimensional datasets. They work well as part of modern ensemble methods like Random Forests, where randomness enhances the ensemble's predictive power. ID3 forests, while foundational, are less commonly used in practice due to their propensity to overfit and higher computational cost.s